# Autoencoder for Credit Card Fraud Detection

This notebook implements an autoencoder for anomaly detection on the Credit Card Fraud Detection dataset. The autoencoder is trained on normal transactions to learn their patterns, and high reconstruction errors are used to flag fraud transactions.

In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns


In [2]:
# Load the sampled data
X_sampled = pd.read_csv('X_sampled.csv')
y_sampled = np.load('y_sampled.npy')

print("Sampled dataset shape:", X_sampled.shape)
print("Sampled class distribution:")
print(pd.Series(y_sampled).value_counts())


Sampled dataset shape: (28481, 29)
Sampled class distribution:
0    28432
1       49
Name: count, dtype: int64


In [3]:
# Split the data into training and test sets
# First, separate normal and fraud transactions
normal_indices = np.where(y_sampled == 0)[0]
fraud_indices = np.where(y_sampled == 1)[0]

X_normal = X_sampled.iloc[normal_indices]
X_fraud = X_sampled.iloc[fraud_indices]
y_normal = y_sampled[normal_indices]
y_fraud = y_sampled[fraud_indices]

# Split normal transactions into train and test (80% train, 20% test)
X_normal_train, X_normal_test, y_normal_train, y_normal_test = train_test_split(
    X_normal, y_normal, test_size=0.2, random_state=42
)

# Combine the normal test set with all fraud transactions for testing
X_test = pd.concat([pd.DataFrame(X_normal_test, columns=X_sampled.columns), X_fraud])
y_test = np.concatenate([y_normal_test, y_fraud])

print("Training set shape (normal only):", X_normal_train.shape)
print("Test set shape (normal + fraud):", X_test.shape)
print("Test set class distribution:")
print(pd.Series(y_test).value_counts())


Training set shape (normal only): (22745, 29)
Test set shape (normal + fraud): (5736, 29)
Test set class distribution:
0    5687
1      49
Name: count, dtype: int64


In [4]:
# Build the autoencoder model
input_dim = X_sampled.shape[1]  # 29 features
encoding_dim = 14  # Compress to half the input size

input_layer = layers.Input(shape=(input_dim,))
encoder = layers.Dense(encoding_dim, activation='relu')(input_layer)
decoder = layers.Dense(input_dim, activation='linear')(encoder)

autoencoder = models.Model(inputs=input_layer, outputs=decoder)
autoencoder.compile(optimizer='adam', loss='mse')

autoencoder.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 29)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 14)             │           420 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 29)             │           435 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 855 (3.34 KB)

 Trainable params: 855 (3.34 KB)

 Non-trainable params: 0 (0.00 B)

In [5]:
# Train the autoencoder on normal transactions
history = autoencoder.fit(
    X_normal_train,
    X_normal_train,
    epochs=50,
    batch_size=256,
    shuffle=True,
    validation_split=0.2,
    verbose=1
)

# Plot training and validation loss
plt.figure(figsize=(10, 6))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Autoencoder Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss (MSE)')
plt.legend()
plt.savefig('autoencoder_training_loss.png')
plt.close()

print("Training complete. Loss plot saved as 'autoencoder_training_loss.png'.")


Epoch 1/50
72/72 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step - loss: 1.2888 - val_loss: 1.1816
Epoch 2/50
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.9475 - val_loss: 1.0113
Epoch 3/50
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.8177 - val_loss: 0.9035
Epoch 4/50
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.7567 - val_loss: 0.8221
Epoch 5/50
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.6867 - val_loss: 0.7611
Epoch 6/50
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.6089 - val_loss: 0.7136
Epoch 7/50
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.5933 - val_loss: 0.6816
Epoch 8/50
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.5522 - val_loss: 0.6589
Epoch 9/50
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.5298 - val_loss: 0.6424
Epoch 10/50
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.5285 - val_loss: 0.6297
Epoch 11/50
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.5196 - val_loss: 0.6212
Epoch 12/50
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.5011 - val_l

In [6]:
# Predict reconstruction errors on the test set
reconstructions = autoencoder.predict(X_test)
mse = np.mean(np.power(X_test - reconstructions, 2), axis=1)

# Plot the distribution of reconstruction errors
plt.figure(figsize=(10, 6))
sns.histplot(mse[y_test == 0], bins=50, color='blue', label='Normal', alpha=0.5)
sns.histplot(mse[y_test == 1], bins=50, color='red', label='Fraud', alpha=0.5)
plt.title('Reconstruction Error Distribution (Normal vs Fraud)')
plt.xlabel('Mean Squared Error')
plt.ylabel('Frequency')
plt.legend()
plt.savefig('reconstruction_error_distribution.png')
plt.close()


180/180 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step


In [7]:
# Determine a threshold for anomaly detection
# Use the 95th percentile of reconstruction errors on normal transactions in the training set
reconstructions_train = autoencoder.predict(X_normal_train)
mse_train = np.mean(np.power(X_normal_train - reconstructions_train, 2), axis=1)
threshold = np.percentile(mse_train, 95)

print("Reconstruction error threshold (95th percentile of normal training data):", threshold)

# Classify test transactions as normal (0) or fraud (1) based on the threshold
autoencoder_labels = np.where(mse > threshold, 1, 0)


711/711 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
Reconstruction error threshold (95th percentile of normal training data): 0.9118323988372491


In [8]:
# Evaluate the autoencoder's performance
print("\nAutoencoder Results on Test Data:")
print(classification_report(y_test, autoencoder_labels))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, autoencoder_labels))



Autoencoder Results on Test Data:
              precision    recall  f1-score   support

           0       1.00      0.95      0.97      5687
           1       0.13      0.90      0.23        49

    accuracy                           0.95      5736
   macro avg       0.56      0.92      0.60      5736
weighted avg       0.99      0.95      0.97      5736


Confusion Matrix:
[[5392  295]
 [   5   44]]


In [9]:
# Plot the number of normal vs. anomalous transactions
anomaly_count = np.sum(autoencoder_labels == 1)
normal_count = np.sum(autoencoder_labels == 0)
plt.figure(figsize=(8, 6))
sns.barplot(x=['Normal', 'Anomaly'], y=[normal_count, anomaly_count], hue=['Normal', 'Anomaly'], legend=False)
plt.title('Autoencoder: Number of Normal vs Anomalous Transactions (Test Data)')
plt.ylabel('Count')
plt.savefig('autoencoder_anomaly_counts.png')
plt.close()

print("Autoencoder anomaly detection complete. Results plot saved as 'autoencoder_anomaly_counts.png'.")


Autoencoder anomaly detection complete. Results plot saved as 'autoencoder_anomaly_counts.png'.
